In [1]:
!rm -rf /kaggle/working/output_videos/*
print("✅ output_videos 文件夹下的旧文件已全部清空！")

✅ output_videos 文件夹下的旧文件已全部清空！


In [ ]:
# ============================================================
# 【Telegram 机器人通知设置】
# 如果不需要通知，保留为空或 False 即可
# ============================================================
ENABLE_TG_BOT = True
TG_BOT_TOKEN = ""
TG_CHAT_ID = ""
# ============================================================

# ============================================================
# 【在这里修改换脸任务配对】
# 格式: "视频文件名" -> "对应的人脸图片文件名"
# ============================================================
EXPLICIT_MAPPING = {
    "test.mp4": "test.jpg",
}
# ============================================================

import os
import sys
import glob
import time
import subprocess
import importlib.util
from pathlib import Path
import cv2

import urllib.request
import urllib.parse
import json

# ============================================================
# STEP 0: Telegram 通知函数
# ============================================================
def send_tg_message(text):
    if not ENABLE_TG_BOT or not TG_BOT_TOKEN or not TG_CHAT_ID:
        return
    try:
        url = f"https://api.telegram.org/bot{TG_BOT_TOKEN}/sendMessage"
        data = urllib.parse.urlencode({'chat_id': TG_CHAT_ID, 'text': text, 'parse_mode': 'HTML'}).encode('utf-8')
        req = urllib.request.Request(url, data=data)
        with urllib.request.urlopen(req, timeout=10) as response:
            pass
    except Exception as e:
        print(f"⚠️ TG 消息推送失败: {e}")

# ============================================================
# STEP 1: 安装依赖
# ============================================================
print("=" * 60)
print("STEP 1: 安装依赖")
print("=" * 60)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "av", "opencv-python", "tinyface"
], check=False)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "onnxruntime-gpu",
    "--extra-index-url",
    "https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/"
], check=False)

import onnxruntime as ort
print(f"ORT 版本: {ort.__version__}")
print(f"可用 Provider: {ort.get_available_providers()}")

# ============================================================
# STEP 2: 检查 GPU 显存
# ============================================================
print("\n" + "=" * 60)
print("STEP 2: GPU 显存状态")
print("=" * 60)
os.system("nvidia-smi --query-gpu=name,memory.used,memory.free,memory.total --format=csv,noheader")

# ============================================================
# STEP 3: 克隆项目代码
# ============================================================
print("\n" + "=" * 60)
print("STEP 3: 检查项目代码")
print("=" * 60)

REPO_DIR = "/kaggle/working/Magic-Mirror"
SRC_DIR  = f"{REPO_DIR}/src-python"

if not os.path.exists(f"{SRC_DIR}/magic/face.py"):
    print("正在克隆项目...")
    ret = os.system(f"git clone -q https://github.com/keggin-CHN/Magic-Mirror.git {REPO_DIR}")
    if ret != 0:
        raise RuntimeError("克隆失败，请检查网络连接！")
    print("克隆完成")
else:
    print(f"项目已存在: {SRC_DIR}")

# ============================================================
# STEP 4: 下载 ONNX 模型文件（每次重启内核都需要重新下载）
# ============================================================
print("\n" + "=" * 60)
print("STEP 4: 检查并下载模型文件")
print("=" * 60)

MODELS_DIR = f"{SRC_DIR}/models"
os.makedirs(MODELS_DIR, exist_ok=True)

MODELS = [
    "scrfd_2.5g.onnx",
    "arcface_w600k_r50.onnx",
    "inswapper_128_fp16.onnx",
    "gfpgan_1.4.onnx",
]

BASE_URL = "https://github.com/keggin-CHN/Magic-Mirror/releases/download/v2.0.0"

for m in MODELS:
    dst = os.path.join(MODELS_DIR, m)
    if os.path.exists(dst) and os.path.getsize(dst) > 1024 * 1024:
        size_mb = os.path.getsize(dst) // 1024 // 1024
        print(f"   已存在: {m}  ({size_mb} MB)")
    else:
        print(f"   下载中: {m} ...")
        url = f"{BASE_URL}/{m}"
        ret = os.system(f"wget -q -c '{url}' -O '{dst}'")
        if ret == 0 and os.path.exists(dst):
            size_mb = os.path.getsize(dst) // 1024 // 1024
            print(f"   下载完成: {m}  ({size_mb} MB)")
        else:
            raise RuntimeError(f"模型下载失败: {m}，URL: {url}")

print(f"\n全部 {len(MODELS)} 个模型就绪！")

# ============================================================
# STEP 5: 载入核心换脸模块
# ============================================================
print("\n" + "=" * 60)
print("STEP 5: 载入换脸核心模块")
print("=" * 60)

face_py = f"{SRC_DIR}/magic/face.py"
if not os.path.exists(face_py):
    raise RuntimeError(f"找不到 face.py: {face_py}")

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

spec = importlib.util.spec_from_file_location("magic.face", face_py)
face_module = importlib.util.module_from_spec(spec)
sys.modules["magic.face"] = face_module
spec.loader.exec_module(face_module)

swap_face_video = face_module.swap_face_video
load_models     = face_module.load_models

print(f"模块载入成功: {face_py}")

# ============================================================
# STEP 6: 重定向输出路径（/kaggle/input 是只读的）
# ============================================================
OUTPUT_DIR = "/kaggle/working/output_videos"
os.makedirs(OUTPUT_DIR, exist_ok=True)
face_module._get_output_video_path = lambda f: f"{OUTPUT_DIR}/swapped_{Path(f).stem}.mp4"
print(f"输出目录: {OUTPUT_DIR}")

# ============================================================
# STEP 7: 初始化模型（加载到内存）
# ============================================================
print("\n" + "=" * 60)
print("STEP 7: 初始化模型")
print("=" * 60)
load_models()
print("模型初始化完成！")

# ============================================================
# STEP 8: 定义换脸任务
# 格式: "视频文件名" -> "对应的人脸图片文件名"
# ============================================================
print("\n" + "=" * 60)
print("STEP 8: 配置换脸任务")
print("=" * 60)

def find_file(name):
    """在 /kaggle/input 下递归查找文件"""
    for p in glob.glob(f"/kaggle/input/**/{name}", recursive=True):
        return p
    return None

def get_video_info(video_path):
    """获取视频的 FPS、尺寸和总帧数"""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return "未知", "未知", "未知"
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return fps, f"{width}x{height}", frame_count

tasks = []
missing_files = []

for video_name, face_name in EXPLICIT_MAPPING.items():
    video_path = find_file(video_name)
    face_path  = find_file(face_name)
    if video_path and face_path:
        tasks.append({
            "vname": video_name,
            "vpath": video_path,
            "fname": face_name,
            "fpath": face_path,
        })
    else:
        missing_files.append(f"  {video_name}: 视频={video_path}, 照片={face_path}")

print(f"\n{'='*65}")
print("换脸任务核对列表：")
print(f"{'='*65}")
for t in tasks:
    print(f"  [{t['vname']}]  <====  [{t['fname']}]")
    print(f"      视频: {t['vpath']}")
    print(f"      人脸: {t['fpath']}")
print(f"{'='*65}")

if missing_files:
    print("以下任务因文件缺失被跳过：")
    for m in missing_files:
        print(m)

if not tasks:
    raise RuntimeError("没有有效任务！请检查 /kaggle/input 下是否存在对应文件。")

print(f"\n共 {len(tasks)} 组任务，开始 GPU 换脸...\n")

# ============================================================
# STEP 9: 依次 GPU 换脸执行
# ============================================================
print("=" * 65)
print("STEP 9: 开始 GPU 批量换脸")
print("=" * 65)

total_t0 = time.time()
results   = []

for i, t in enumerate(tasks, 1):
    print(f"\n[{i}/{len(tasks)}]  {t['vname']}  <=  {t['fname']}")
    t0 = time.time()
    try:
        # 获取视频信息用于推送
        fps, resolution, total_frames = get_video_info(t["vpath"])
        
        out = swap_face_video(
            input_path   = t["vpath"],
            face_path    = t["fpath"],
            use_gpu      = True,
            gpu_provider = "cuda",
        )
        
        elapsed = time.time() - t0
        print(f"   ✅ 成功！耗时 {elapsed:.0f}s")
        print(f"   输出: {out}")
        results.append({"name": t["vname"], "ok": True, "out": out, "time": elapsed})
        
        # 计算输出文件大小
        out_size_mb = os.path.getsize(out) / (1024 * 1024) if os.path.exists(out) else 0.0
        
        # 发送成功通知（一行一个）
        msg = (
            f"✅ <b>任务完成 [{i}/{len(tasks)}]</b>\n"
            f"🎬 视频: <code>{t['vname']}</code>\n"
            f"👤 人脸: <code>{t['fname']}</code>\n"
            f"📦 大小: <b>{out_size_mb:.2f} MB</b>\n"
            f"⚡️ FPS: <b>{fps}</b>\n"
            f"🎥 尺寸: <b>{resolution}</b>\n"
            f"🎞 帧数: <b>{total_frames}</b>\n"
            f"⏱ 耗时: <b>{int(elapsed//60)}分{int(elapsed%60)}秒</b>\n"
            f"📈 总进度: <b>{i}/{len(tasks)}</b>"
        )
        send_tg_message(msg)

    except Exception as e:
        elapsed = time.time() - t0
        print(f"   失败！耗时 {elapsed:.0f}s")
        print(f"   原因: {e}")
        results.append({"name": t["vname"], "ok": False, "err": str(e), "time": elapsed})
        
        # 发送失败通知（一行一个）
        msg = (
            f"❌ <b>任务失败 [{i}/{len(tasks)}]</b>\n"
            f"🎬 视频: <code>{t['vname']}</code>\n"
            f"👤 人脸: <code>{t['fname']}</code>\n"
            f"⚠️ 错误信息:\n<code>{str(e)[:200]}</code>"
        )
        send_tg_message(msg)

# ============================================================
# STEP 10: 汇总结果
# ============================================================
total_elapsed = time.time() - total_t0
ok_count   = sum(1 for r in results if r["ok"])
fail_count = len(results) - ok_count

# 发送最终汇总通知
final_msg = (
    f"🎉 <b>所有换脸任务运行结束！</b>\n\n"
    f"✅ 成功: {ok_count} 个\n"
    f"❌ 失败: {fail_count} 个\n"
    f"⏱ 总耗时: <b>{int(total_elapsed//60)}分 {int(total_elapsed%60)}秒</b>"
)
send_tg_message(final_msg)

print(f"\n{'='*65}")
print(f"全部完成！总耗时: {total_elapsed:.0f} 秒")
print(f"{'='*65}")
print(f"  成功: {ok_count} 个")
print(f"  失败: {fail_count} 个")
print(f"\n输出文件列表 ({OUTPUT_DIR}):")
for r in results:
    if r["ok"]:
        print(f"  OK  {r['name']} -> {r['out']}  ({r['time']:.0f}s)")
    else:
        print(f"  NG  {r['name']} -> 失败: {r['err'][:100]}")
print("=" * 65)

STEP 1: 安装依赖
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 56.4 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.5.2 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.2 which is incompatible.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.4/475.4 MB 3.7 MB/s eta 0:00:00
ORT 版本: 1.29.0
可用 Provider: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']

STEP 2: GPU 显存状态
Tesla P100-PCIE-16GB, 0 MiB, 16270 MiB, 16384 MiB

STEP 3: 检查项目代码
正在克隆项目...
克隆完成

STEP 4: 检查并下载模型文件
   下载中: scrfd_2.5g.onnx ...
   下载完成: scrfd_2.5g.onnx  (3 MB)
   下载中: arcface_w600k_r50.onnx ...
   下载完成: arcface_w600k_r50.onnx  (166 MB)
   下载中: inswapper_128_fp16.onnx ...
   下载完成: inswapper_128_fp16.onnx  (264 MB)
   下载中: gfpgan_1.4.onnx ...
   下载完成: gfpgan_1.4.onnx  (324 MB)

全部 4 个模型就绪！

STEP 5: 载入换脸核心模块
模块载入成功: /kaggle/working/Magic-Mirror/src-python/magic/face.py
输出目录: /kaggle/working/output_videos

STEP 7: 初始化模型
模型初始化完成！

STEP 8: 配置换脸任务

换脸任务核对列表：
  [test.mp4]  <====  [test.jpg]
      视频: /kaggle/input/models/keggin/1/tensorflow2/default/10/test.mp4
      人脸: /kaggle/input/models/keggin/2/tensorflow2/default/6/test.jpg

共 1 组任务，开始 GPU 换脸...

STEP 9: 开始 GPU 批量换脸

[1/1]  test.mp4